# Model Development & Experiment Tracking

## Objective

Develop and evaluate machine learning models for retail sales forecasting using the feature-engineered dataset.

The primary goals of this notebook are:

1. Establish a reliable time-series validation framework.
2. Build a baseline sales forecasting model.
3. Compare multiple machine learning algorithms.
4. Evaluate the impact of customer information on sales prediction.
5. Track experiments and model performance for reproducibility.

## Modeling Strategy

### Experiment 1: Direct Sales Forecasting

Predict store sales using historical sales, promotional, competition, calendar and store-level features without using customer counts.

### Experiment 2: Customer Demand Forecasting

Predict customer footfall using engineered features.

### Experiment 3: Two-Stage Demand Intelligence Pipeline

Predict customers first and then use predicted customer demand to improve sales forecasting.

## Evaluation Principles

* Time-based train-validation split will be used.
* Future information must never leak into training data.
* Models will be compared using forecasting performance metrics.
* Feature importance analysis will be performed to understand business drivers of demand.


In [2]:
# Load model-ready dataset

import pandas as pd

df = pd.read_csv(
    "../data/processed/model_ready_data.csv"
)

print(df.shape)
df.head()

(985989, 42)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,SchoolHoliday,CompetitionDistance,Promo2,...,StateHoliday_c,Sales_Lag_7,Sales_Lag_14,Sales_Lag_28,Sales_RollingMean_7,Sales_RollingMean_14,Sales_RollingMean_28,Sales_RollingStd_7,Sales_RollingStd_14,Sales_RollingStd_28
0,1,2,2013-01-29,3725,522,1,0,0,1270.0,0,...,0,5720.0,3900.0,0.0,4533.142857,4170.500000,4121.285714,2080.017823,1899.595898,2075.631146
1,1,3,2013-01-30,4601,560,1,0,0,1270.0,0,...,0,5578.0,4008.0,5530.0,4248.142857,4158.000000,4254.321429,2026.274696,1902.086951,1914.845455
2,1,4,2013-01-31,4709,571,1,0,0,1270.0,0,...,0,5195.0,4044.0,4327.0,4108.571429,4200.357143,4221.142857,1951.681400,1905.090008,1899.913267
3,1,5,2013-02-01,5633,658,1,0,0,1270.0,0,...,0,5586.0,4127.0,4486.0,4039.142857,4247.857143,4234.785714,1914.889329,1909.177546,1902.071860
4,1,6,2013-02-02,5970,701,1,0,0,1270.0,0,...,0,5598.0,5182.0,4997.0,4045.857143,4355.428571,4275.750000,1921.288841,1943.954681,1919.949818


In [3]:
# Verify forecasting timeline

print(df["Date"].min())
print(df["Date"].max())

2013-01-29
2015-07-31


In [4]:
df.columns.tolist()

['Store',
 'DayOfWeek',
 'Date',
 'Sales',
 'Customers',
 'Open',
 'Promo',
 'SchoolHoliday',
 'CompetitionDistance',
 'Promo2',
 'Year',
 'Month',
 'Quarter',
 'Day',
 'WeekOfYear',
 'IsMonthStart',
 'IsMonthEnd',
 'IsQuarterStart',
 'IsQuarterEnd',
 'IsWeekend',
 'CompetitionAgeMonths',
 'CompetitionDateKnown',
 'CompetitionActive',
 'Promo2AgeMonths',
 'Promo2Active',
 'StoreType_b',
 'StoreType_c',
 'StoreType_d',
 'Assortment_b',
 'Assortment_c',
 'StateHoliday_a',
 'StateHoliday_b',
 'StateHoliday_c',
 'Sales_Lag_7',
 'Sales_Lag_14',
 'Sales_Lag_28',
 'Sales_RollingMean_7',
 'Sales_RollingMean_14',
 'Sales_RollingMean_28',
 'Sales_RollingStd_7',
 'Sales_RollingStd_14',
 'Sales_RollingStd_28']

## Time-Based Train Validation Split

Train:
2013-01-29 → 2015-05-31

Validation:
2015-06-01 → 2015-07-31

This gives:

~28 months training

~2 months validation



In [5]:
# Create time-based train and validation datasets

split_date = "2015-06-01"

train_df = df[df["Date"] < split_date].copy()
valid_df = df[df["Date"] >= split_date].copy()

print("Train Shape :", train_df.shape)
print("Validation Shape :", valid_df.shape)

print("\nTrain Period:")
print(train_df["Date"].min(), "to", train_df["Date"].max())

print("\nValidation Period:")
print(valid_df["Date"].min(), "to", valid_df["Date"].max())

Train Shape : (917974, 42)
Validation Shape : (68015, 42)

Train Period:
2013-01-29 to 2015-05-31

Validation Period:
2015-06-01 to 2015-07-31


## Time-Based Validation Strategy

A chronological train-validation split was used instead of a random split to simulate real-world forecasting conditions.

### Training Period

* 2013-01-29 to 2015-05-31

### Validation Period

* 2015-06-01 to 2015-07-31

### Rationale

In forecasting problems, future observations must remain unseen during training. Random splitting can introduce temporal leakage by allowing information from future periods to influence model training.

Using a time-based split ensures that model evaluation closely resembles real-world deployment, where predictions are generated for future dates using only historical information.


## Experiment 1: Direct Sales Forecasting

## Experiment 1: Direct Sales Forecasting

### Objective

Predict future store sales using only information that would realistically be available at prediction time.

### Target Variable

* Sales

### Excluded Features

#### Customers

Customer counts are strongly correlated with sales but are unknown for future dates. Including them would create an unrealistic forecasting setup.

#### Date

Raw date information was excluded because relevant temporal patterns have already been captured through engineered calendar features.

### Included Features

The model uses store-level, promotional, competition, calendar, lag and rolling-window features to forecast future sales.

### Business Rationale

This experiment represents a production-ready forecasting scenario where future customer demand is unavailable and sales must be predicted directly from historical business information.


In [6]:
# Experiment 1: Direct Sales Forecasting

TARGET = "Sales"

drop_cols = [
    "Sales",
    "Customers",
    "Date"
]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df[TARGET]

X_valid = valid_df.drop(columns=drop_cols)
y_valid = valid_df[TARGET]

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

X_train: (917974, 39)
X_valid: (68015, 39)
y_train: (917974,)
y_valid: (68015,)


## Baseline Model

### Objective

Establish a minimum performance benchmark before training advanced machine learning models.

### Why a Baseline?

Model performance should always be evaluated relative to a simple reference model. Without a baseline, it is difficult to determine whether a complex model is providing meaningful improvements.

### Dummy Regressor

A Dummy Regressor ignores all input features and predicts a constant value based on the training target distribution.

For this project, the baseline model predicts the average sales value observed in the training dataset.

### Success Criterion

Any machine learning model developed later must outperform the baseline model by a significant margin to justify its complexity.


In [7]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

In [8]:
# Baseline Model

dummy_model = DummyRegressor(strategy="mean")

dummy_model.fit(X_train, y_train)

dummy_preds = dummy_model.predict(X_valid)

In [9]:
# Baseline Metrics

mae = mean_absolute_error(y_valid, dummy_preds)

rmse = np.sqrt(
    mean_squared_error(y_valid, dummy_preds)
)

r2 = r2_score(y_valid, dummy_preds)

print("Baseline Model Results")
print("-" * 30)
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

Baseline Model Results
------------------------------
MAE  : 2830.86
RMSE : 3822.92
R²   : -0.0114


## Baseline Model Evaluation

### Results

* MAE: 2830.86
* RMSE: 3822.92
* R²: -0.0114

### Key Findings

The baseline model performed poorly because it predicts a constant average sales value for all observations and ignores all available business information.

The negative R² score indicates that the model explains virtually none of the variation in sales and performs slightly worse than a simple mean prediction benchmark.

### Conclusion

The poor baseline performance confirms that sales are influenced by multiple factors including promotions, store characteristics, competition, seasonality and historical demand patterns. Therefore, machine learning models have significant potential to improve forecasting accuracy.


In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_valid)

rf_mae = mean_absolute_error(y_valid, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_valid, rf_preds))
rf_r2 = r2_score(y_valid, rf_preds)

print("Random Forest Results")
print("-"*30)
print(f"MAE  : {rf_mae:.2f}")
print(f"RMSE : {rf_rmse:.2f}")
print(f"R²   : {rf_r2:.4f}")

Random Forest Results
------------------------------
MAE  : 576.15
RMSE : 882.68
R²   : 0.9461


## Random Forest Evaluation

### Results

* MAE: 576.15
* RMSE: 882.68
* R²: 0.9461

### Comparison with Baseline

The Random Forest model significantly outperformed the Dummy Regressor across all evaluation metrics.

| Metric | Baseline | Random Forest |
| ------ | -------: | ------------: |
| MAE    |  2830.86 |        576.15 |
| RMSE   |  3822.92 |        882.68 |
| R²     |  -0.0114 |        0.9461 |

### Key Findings

* Historical lag features and rolling statistics contributed substantial predictive power.
* The model successfully captured non-linear relationships between sales, promotions, store characteristics and temporal patterns.
* Feature engineering resulted in a dramatic reduction in forecasting error relative to the baseline model.

### Conclusion

Random Forest establishes a strong benchmark model and demonstrates that the engineered feature set contains significant predictive signal for retail demand forecasting.


In [11]:
# Feature Importance

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(20)

,Feature,Importance
2,Open,0.478239
31,Sales_Lag_14,0.281050
32,Sales_Lag_28,0.060648
35,Sales_RollingMean_28,0.046271
3,Promo,0.034241
36,Sales_RollingStd_7,0.024767
33,Sales_RollingMean_7,0.018792
10,Day,0.011072
1,DayOfWeek,0.010262
11,WeekOfYear,0.008829


In [12]:
(
    (df["Open"] == 0) &
    (df["Sales"] == 0)
).sum()

np.int64(167144)

In [14]:
df["Open"].value_counts()

Open
1    818845
0    167144
Name: count, dtype: int64

## Feature Importance Analysis

### Key Findings

The Random Forest model identified historical sales behaviour as the strongest predictor of future sales.

### Most Important Features

1. Open
2. Sales_Lag_14
3. Sales_Lag_28
4. Sales_RollingMean_28
5. Promo

### Business Insights

* Store operational status has the strongest influence on sales.
* Historical sales patterns provide substantial predictive signal.
* Promotional activity remains an important business driver.
* Rolling statistics successfully capture underlying demand trends.
* Competition-related features contribute relatively little additional information once historical sales behaviour is available.

### Conclusion

The feature importance analysis validates the effectiveness of the feature engineering process and confirms that lag and rolling-window features are critical components of the forecasting pipeline.


## XGBoost Regressor

### Objective

Train a gradient boosting model capable of learning complex non-linear relationships and sequentially correcting prediction errors.

### Why XGBoost?

* State-of-the-art performance on structured/tabular data.
* Handles non-linear interactions effectively.
* Built-in regularization reduces overfitting.
* Frequently used in retail forecasting competitions and production systems.
* Often outperforms bagging-based methods such as Random Forest.

### Success Criterion

The XGBoost model should outperform the Random Forest benchmark in terms of RMSE and R² score while maintaining good generalization on unseen validation data.


In [15]:
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_valid)

xgb_mae = mean_absolute_error(y_valid, xgb_preds)
xgb_rmse = np.sqrt(mean_squared_error(y_valid, xgb_preds))
xgb_r2 = r2_score(y_valid, xgb_preds)

print("XGBoost Results")
print("-" * 30)
print(f"MAE  : {xgb_mae:.2f}")
print(f"RMSE : {xgb_rmse:.2f}")
print(f"R²   : {xgb_r2:.4f}")

XGBoost Results
------------------------------
MAE  : 539.19
RMSE : 816.03
R²   : 0.9539


In [16]:
xgb_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgb_model.feature_importances_
})

xgb_importance = xgb_importance.sort_values(
    by="Importance",
    ascending=False
)

xgb_importance.head(20)

,Feature,Importance
2,Open,0.557268
31,Sales_Lag_14,0.093816
29,StateHoliday_c,0.059406
27,StateHoliday_a,0.045943
35,Sales_RollingMean_28,0.043305
3,Promo,0.040602
28,StateHoliday_b,0.028137
1,DayOfWeek,0.015727
32,Sales_Lag_28,0.015570
33,Sales_RollingMean_7,0.011699


## XGBoost Evaluation

### Results

* MAE: 539.19
* RMSE: 816.03
* R²: 0.9539

### Comparison with Previous Models

| Model           |     MAE |    RMSE |      R² |
| --------------- | ------: | ------: | ------: |
| Dummy Regressor | 2830.86 | 3822.92 | -0.0114 |
| Random Forest   |  576.15 |  882.68 |  0.9461 |
| XGBoost         |  539.19 |  816.03 |  0.9539 |

### Key Findings

* XGBoost outperformed Random Forest across all evaluation metrics.
* The model effectively leveraged lag features, rolling statistics, promotional information and temporal patterns.
* Gradient boosting successfully captured complex non-linear relationships within the retail sales data.

### Conclusion

XGBoost establishes a new benchmark for the project and demonstrates strong forecasting performance on unseen validation data.


## XGBoost Feature Importance Analysis

### Key Findings

The XGBoost model identified operational status, promotional activity, holidays and historical sales behaviour as the primary drivers of sales.

### Most Important Features

* Open
* Sales_Lag_14
* StateHoliday indicators
* Sales_RollingMean_28
* Promo

### Business Insights

* Store closures have a substantial impact on overall sales.
* Promotional campaigns significantly influence demand.
* State holidays introduce meaningful shifts in customer purchasing behaviour.
* Historical sales patterns remain strong predictors of future demand.

### Conclusion

The feature importance analysis confirms that sales forecasting performance is driven by a combination of operational status, calendar effects, promotions and historical demand trends.


## LightGBM Regressor

### Objective

Train a highly optimized gradient boosting model designed for large-scale tabular datasets and compare its performance against Random Forest and XGBoost.

### Why LightGBM?

* Faster training on large datasets.
* Efficient memory utilization.
* Handles high-dimensional feature spaces effectively.
* Often achieves performance comparable to or better than XGBoost.
* Widely used in production-grade forecasting systems.

### Success Criterion

The LightGBM model should match or outperform the XGBoost benchmark while maintaining strong generalization performance on the validation dataset.


In [17]:
from lightgbm import LGBMRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(X_train, y_train)

lgbm_preds = lgbm_model.predict(X_valid)

lgbm_mae = mean_absolute_error(y_valid, lgbm_preds)
lgbm_rmse = np.sqrt(mean_squared_error(y_valid, lgbm_preds))
lgbm_r2 = r2_score(y_valid, lgbm_preds)

print("LightGBM Results")
print("-" * 30)
print(f"MAE  : {lgbm_mae:.2f}")
print(f"RMSE : {lgbm_rmse:.2f}")
print(f"R²   : {lgbm_r2:.4f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.038052 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3307
[LightGBM] [Info] Number of data points in the train set: 917974, number of used features: 39
[LightGBM] [Info] Start training from score 5764.640052
LightGBM Results
------------------------------
MAE  : 555.05
RMSE : 836.76
R²   : 0.9515


## Model Comparison and Selection

### Final Results

| Model           |     MAE |    RMSE |      R² |
| --------------- | ------: | ------: | ------: |
| Dummy Regressor | 2830.86 | 3822.92 | -0.0114 |
| Random Forest   |  576.15 |  882.68 |  0.9461 |
| XGBoost         |  539.19 |  816.03 |  0.9539 |
| LightGBM        |  555.05 |  836.76 |  0.9515 |

### Key Findings

* All machine learning models significantly outperformed the baseline benchmark.
* Gradient boosting methods achieved the strongest forecasting performance.
* XGBoost delivered the best overall results on the validation set.
* LightGBM achieved comparable performance while training efficiently on a large dataset.
* Feature engineering contributed substantially to model performance improvements.

### Selected Model

Based on validation performance, XGBoost was selected as the primary forecasting model for subsequent experiments and deployment.

### Conclusion

The combination of lag features, rolling statistics, promotional information and calendar-based features enabled highly accurate retail demand forecasting, with XGBoost achieving the strongest generalization performance.


In [18]:
import joblib

joblib.dump(
    xgb_model,
    "../models/xgboost_sales_model.pkl"
)

['../models/xgboost_sales_model.pkl']